In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from pathlib import Path

# Set up plotting
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
data_path = Path('../data/raw/Full CFPB Dataset')
df = pd.read_csv(data_path, encoding='utf-8')
 
print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")

In [ ]:
print("\n=== Product Distribution ===")
product_counts = df['Product'].value_counts()
print(product_counts)
 
plt.figure(figsize=(12, 6))
product_counts.plot(kind='bar')
plt.title('Distribution of Complaints by Product')
plt.xlabel('Product')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()
 

In [ ]:
df['narrative_length'] = df['Consumer complaint narrative'].fillna('').str.split().str.len()
 
print("\n=== Narrative Length Statistics ===")
print(df['narrative_length'].describe())
 
plt.figure(figsize=(12, 6))
plt.hist(df['narrative_length'], bins=50, edgecolor='black')
plt.title('Distribution of Consumer Narrative Length (Word Count)')
plt.xlabel('Word Count')
plt.ylabel('Frequency')
plt.axvline(df['narrative_length'].median(), color='red', linestyle='--', label=f'Median: {df["narrative_length"].median():.0f}')
plt.legend()
plt.show()
 

In [ ]:
has_narrative = df['Consumer complaint narrative'].notna().sum()
no_narrative = df['Consumer complaint narrative'].isna().sum()
 
print(f"\n=== Narrative Availability ===")
print(f"With narrative: {has_narrative} ({has_narrative/len(df)*100:.1f}%)")
print(f"Without narrative: {no_narrative} ({no_narrative/len(df)*100:.1f}%)")
 

In [ ]:
target_products = ['Credit card', 'Credit card or prepaid card', 'Personal loan', 
                   'Savings account', 'Money transfer']
 
# Normalize product names for filtering
df_filtered = df[df['Product'].isin(target_products)].copy()
 
print(f"\n=== After Filtering for Target Products ===")
print(f"Records: {len(df_filtered)}")
print(df_filtered['Product'].value_counts())
 

In [ ]:
df_filtered = df_filtered[df_filtered['Consumer complaint narrative'].notna()].copy()
df_filtered = df_filtered[df_filtered['Consumer complaint narrative'].str.strip() != ''].copy()
 
print(f"\n=== After Removing Empty Narratives ===")
print(f"Records: {len(df_filtered)}")

In [ ]:
def clean_text(text):
    """Clean consumer complaint narrative."""
    if pd.isna(text) or text == '':
        return ''
    
    # Convert to lowercase
    text = str(text).lower()
    
    # Remove boilerplate text
    boilerplate_phrases = [
        r'i am writing to file a complaint',
        r'this is a complaint',
        r'i would like to file a complaint',
        r'i am submitting this complaint'
    ]
    for phrase in boilerplate_phrases:
        text = re.sub(phrase, '', text, flags=re.IGNORECASE)
    
    # Remove special characters but keep alphanumeric and basic punctuation
    text = re.sub(r'[^\w\s\.,!?;:\'-]', ' ', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text
 
# Apply cleaning
df_filtered['cleaned_narrative'] = df_filtered['Consumer complaint narrative'].apply(clean_text)
 


In [ ]:
# Cell 9: Save filtered and cleaned dataset
output_path = Path('../data/processed/filtered_complaints.csv')
df_filtered.to_csv(output_path, index=False, encoding='utf-8')
 
print(f"\n=== Saved Filtered Dataset ===")
print(f"Location: {output_path}")
print(f"Records: {len(df_filtered)}")
print(f"Columns: {df_filtered.columns.tolist()}")
 

In [ ]:
print("\n=== EDA Summary ===")
print(f"Original dataset size: {len(df)}")
print(f"After product filtering: {len(df[df['Product'].isin(target_products)])}")
print(f"After removing empty narratives: {len(df_filtered)}")
print(f"\nProduct distribution in final dataset:")
print(df_filtered['Product'].value_counts())
print(f"\nAverage narrative length: {df_filtered['narrative_length'].mean():.1f} words")
print(f"Median narrative length: {df_filtered['narrative_length'].median():.1f} words")
